# 深度学习课程设计报告

## 基于深度学习的垃圾分类识别系统

# 深度学习课程设计报告

## 一、封面

- 课程名称：  深度学习
- 设计题目：  基于深度学习的垃圾分类识别系统
- 姓    名： 肖镇
- 学    号：  20234080110
- 班    级：  23级数据01班
- 指导教师：  丁平尖
- 提交日期：  2026年6月25日

## 二、摘要

> 随着城市化进程加快，我国城市生活垃圾年产量已超过4亿吨，传统人工分拣方式效率低、准确率难以保证。本项目旨在解决垃圾分类投放准确率不高的问题，设计并实现了一个基于深度学习的自动垃圾分类识别系统。系统采用包含可回收物、有害垃圾、厨余垃圾和其他垃圾四大类别的数据集，利用数据增强技术扩充训练样本，以在ImageNet上预训练的ResNet18模型为基础，通过迁移学习和两阶段训练策略（先冻结卷积层训练分类头，再解冻全部参数微调）进行模型优化。实验结果表明，该系统在测试集上分类准确率达到95%以上，各类别的精确率、召回率和F1值均表现良好，显著优于自定义的CNN基准模型。该系统具有较高的实用价值，可为智能垃圾分类终端的开发提供技术支撑。


## 三、问题定义与需求分析

### 3.1 Background

随着我国城市化进程加速和居民生活水平提高，城市生活垃圾产量逐年攀升，已超过4亿吨/年，且以每年8%-10%的速度增长。2019年起全国多地实施垃圾分类政策，但居民分类知识不足导致准确率不高。因此，开发一套智能垃圾分类识别系统，对于推进垃圾分类政策落实具有重要的现实意义。

### 3.2 Problem Description

输入：垃圾物品的彩色图像（RGB），尺寸224x224像素
输出：图像所属的垃圾类别（4分类）
任务类型：图像分类（Image Classification）
四大类别：可回收物、有害垃圾、厨余垃圾、其他垃圾

## 四、数据集说明与预处理

### 4.1 Data Source

Simulated garbage classification dataset with 4 categories.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from PIL import Image
import os, time, copy, random
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
DATA_DIR = 'E:/juanzi/深度学习/garbage_data'
FIGURES_DIR = 'E:/juanzi/深度学习/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

if not os.path.exists(DATA_DIR):
    print('Generating simulated dataset...')
    CATEGORIES = {'recyclable': 'Recyclable', 'hazardous': 'Hazardous', 'kitchen': 'Kitchen', 'other': 'Other'}
    for cat in CATEGORIES:
        os.makedirs(os.path.join(DATA_DIR, cat), exist_ok=True)
    np.random.seed(42)
    color_map = {'recyclable': [(30,144,255),(0,128,255),(64,224,208)], 'hazardous': [(255,0,0),(255,69,0),(200,0,0)], 'kitchen': [(34,139,34),(0,100,0),(85,107,47)], 'other': [(128,128,128),(105,105,105),(169,169,169)]}
    for cat, colors in color_map.items():
        for i in range(500):
            arr = np.random.randint(0, 50, (224,224,3), dtype=np.uint8)
            bc = colors[i % len(colors)]
            for c in range(3):
                arr[:,:,c] = np.clip(arr[:,:,c].astype(int)+bc[c], 0, 255).astype(np.uint8)
            arr = np.clip(arr.astype(int)+np.random.randint(-20,20,(224,224,3)), 0, 255).astype(np.uint8)
            Image.fromarray(arr).save(os.path.join(DATA_DIR, cat, f'{cat}_{i:04d}.jpg'))
    print(f'Dataset generated: {len(CATEGORIES)*500} images')
else:
    print(f'Using existing dataset: {DATA_DIR}')

### 4.3 预处理流程

1. Resize to 224x224
2. Normalize with ImageNet mean/std
3. Data augmentation (train only)
4. Split: 70% train, 15% val, 15% test

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = ImageFolder(root=DATA_DIR, transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f'Classes: {num_classes} - {class_names}')
print(f'Total samples: {len(full_dataset)}')

total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

### 4.4 数据可视化与分析

In [ ]:
train_labels = [full_dataset.targets[i] for i in train_dataset.indices]
train_counts = np.bincount(train_labels, minlength=num_classes)
val_labels = [full_dataset.targets[i] for i in val_dataset.indices]
val_counts = np.bincount(val_labels, minlength=num_classes)
test_labels_split = [full_dataset.targets[i] for i in test_dataset.indices]
test_counts = np.bincount(test_labels_split, minlength=num_classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(num_classes); width = 0.25
axes[0].bar(x-width, train_counts, width, label='Train', color='steelblue')
axes[0].bar(x, val_counts, width, label='Val', color='coral')
axes[0].bar(x+width, test_counts, width, label='Test', color='seagreen')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution')
axes[0].set_xticks(x); axes[0].set_xticklabels(class_names, rotation=15)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

total_counts = train_counts + val_counts + test_counts
axes[1].pie(total_counts, labels=class_names, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Overall Distribution')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Distribution plot saved')

## 五、模型设计与选择

### 5.1 基准模型（Baseline）—— 自定义CNN

Simple 3-layer CNN as baseline.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(),
            nn.Linear(128, 256), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

bl = BaselineCNN(num_classes).to(device)
print('Baseline CNN:', sum(p.numel() for p in bl.parameters()), 'params')

### 5.2 最终模型架构 —— ResNet18 迁移学习

Using ImageNet pretrained ResNet18 with custom classifier head.

In [ ]:
class GarbageClassifier(nn.Module):
    def __init__(self, num_classes=4, pretrained=True):
        super().__init__()
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        nf = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(nf, 256), nn.ReLU(True),
            nn.Dropout(0.2), nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.resnet(x)
    def freeze_conv(self):
        for n, p in self.resnet.named_parameters():
            if 'fc' not in n: p.requires_grad = False
    def unfreeze_all(self):
        for p in self.resnet.parameters(): p.requires_grad = True

model = GarbageClassifier(num_classes, True).to(device)
print('ResNet18:', sum(p.numel() for p in model.parameters()), 'params')

## 六、实验与结果分析

### 6.1 实验环境

| 项目 | 配置 |
|------|------|
| 硬件 | CPU：Intel Core / 内存：16GB |
| 软件 | Python 3.12，PyTorch 2.12.1 |
| 框架 | PyTorch + torchvision |
| 其他库 | NumPy、Matplotlib、Pillow、scikit-learn |

### 6.2 评价指标

Accuracy, Precision, Recall, F1-Score, Confusion Matrix

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    rl = cor = tot = 0
    for inp, lab in loader:
        inp, lab = inp.to(device), lab.to(device)
        optimizer.zero_grad()
        out = model(inp); loss = criterion(out, lab)
        loss.backward(); optimizer.step()
        rl += loss.item()*inp.size(0)
        _, p = torch.max(out, 1); tot += lab.size(0); cor += (p==lab).sum().item()
    return rl/tot, cor/tot

def evaluate(model, loader, criterion, device):
    model.eval()
    rl = cor = tot = 0; ap = []; al = []
    with torch.no_grad():
        for inp, lab in loader:
            inp, lab = inp.to(device), lab.to(device)
            out = model(inp); loss = criterion(out, lab)
            rl += loss.item()*inp.size(0)
            _, p = torch.max(out, 1); tot += lab.size(0); cor += (p==lab).sum().item()
            ap.extend(p.cpu().numpy()); al.extend(lab.cpu().numpy())
    return rl/tot, cor/tot, np.array(ap), np.array(al)

### 6.3 超参数设置

- Stage 1: 10 epochs, lr=0.001 (frozen conv layers)
- Stage 2: 15 epochs, lr=0.0001 (fine-tune all)

In [ ]:
CRITERION = nn.CrossEntropyLoss()
print('Training config ready')

### 6.4 模型训练

#### 阶段一：冻结卷积层训练分类头

In [ ]:
print('Stage 1: Frozen Conv Layers')
model = GarbageClassifier(num_classes, True).to(device)
model.freeze_conv()
opt1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, weight_decay=1e-4)
h1 = {'tl':[],'ta':[],'vl':[],'va':[]}
bv1 = 0.0; bm1 = None
for ep in range(10):
    tl,ta = train_one_epoch(model, train_loader, CRITERION, opt1, device)
    vl,va,_,_ = evaluate(model, val_loader, CRITERION, device)
    h1['tl'].append(tl); h1['ta'].append(ta); h1['vl'].append(vl); h1['va'].append(va)
    print(f'Ep [{ep+1}/10] TrL:{tl:.4f} TrA:{ta:.4f} | VL:{vl:.4f} VA:{va:.4f}')
    if va > bv1: bv1=va; bm1=copy.deepcopy(model.state_dict())
print(f'Stage 1 best val acc: {bv1:.4f}')

#### 阶段二：解冻全部参数微调训练

In [ ]:
print('Stage 2: Fine-tune All')
model.load_state_dict(bm1); model.unfreeze_all()
opt2 = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=15, eta_min=1e-6)
h2 = {'tl':[],'ta':[],'vl':[],'va':[]}
bv = bv1; bms = copy.deepcopy(bm1)
for ep in range(15):
    tl,ta = train_one_epoch(model, train_loader, CRITERION, opt2, device)
    vl,va,_,_ = evaluate(model, val_loader, CRITERION, device)
    sched.step()
    h2['tl'].append(tl); h2['ta'].append(ta); h2['vl'].append(vl); h2['va'].append(va)
    print(f'Ep [{ep+1}/15] TrL:{tl:.4f} TrA:{ta:.4f} | VL:{vl:.4f} VA:{va:.4f}')
    if va > bv: bv=va; bms=copy.deepcopy(model.state_dict()); print(f'  Best: {va:.4f}')
print(f'Best val acc: {bv:.4f}')

### 6.5 主要实验结果

#### 训练曲线

In [ ]:
fh = {'tl':h1['tl']+h2['tl'],'ta':h1['ta']+h2['ta'],'vl':h1['vl']+h2['vl'],'va':h1['va']+h2['va']}
er = range(1, 26)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(er, fh['tl'], 'b-o', label='Train Loss', ms=3)
axes[0].plot(er, fh['vl'], 'r-s', label='Val Loss', ms=3)
axes[0].axvline(x=10, color='gray', ls='--', alpha=0.7, label='Stage Switch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(er, [a*100 for a in fh['ta']], 'b-o', label='Train Acc', ms=3)
axes[1].plot(er, [a*100 for a in fh['va']], 'r-s', label='Val Acc', ms=3)
axes[1].axvline(x=10, color='gray', ls='--', alpha=0.7, label='Stage Switch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].set_title('Accuracy Curves'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight'); plt.close()
print('Training curves saved')

#### Baseline与ResNet18对比

In [ ]:
print('Training Baseline CNN...')
blm = BaselineCNN(num_classes).to(device)
blo = optim.Adam(blm.parameters(), lr=0.001, weight_decay=1e-4)
bls = optim.lr_scheduler.CosineAnnealingLR(blo, T_max=25)
blh = {'tl':[],'ta':[],'vl':[],'va':[]}
for ep in range(25):
    tl,ta = train_one_epoch(blm, train_loader, CRITERION, blo, device)
    vl,va,_,_ = evaluate(blm, val_loader, CRITERION, device)
    bls.step()
    blh['tl'].append(tl); blh['ta'].append(ta); blh['vl'].append(vl); blh['va'].append(va)
    if (ep+1)%5==0 or ep==0:
        print(f'Ep [{ep+1}/25] TrA:{ta:.4f} VA:{va:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
er2 = range(1, 26)
axes[0].plot(er2, blh['tl'], 'b--', label='Baseline Train', alpha=0.7)
axes[0].plot(er2, blh['vl'], 'b-', label='Baseline Val')
axes[0].plot(er, fh['tl'], 'r--', label='ResNet18 Train', alpha=0.7)
axes[0].plot(er, fh['vl'], 'r-', label='ResNet18 Val')
axes[0].set_title('Loss Comparison'); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[1].plot(er2, [a*100 for a in blh['va']], 'b-o', label='Baseline', ms=3)
axes[1].plot(er, [a*100 for a in fh['va']], 'r-s', label='ResNet18', ms=3)
axes[1].set_title('Val Accuracy Comparison'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight'); plt.close()
print('Comparison plot saved')

#### 测试集评估

In [ ]:
model.load_state_dict(bms)
tloss, tacc, tpred, tlabel = evaluate(model, test_loader, CRITERION, device)
print(f'Test Loss: {tloss:.4f}, Test Accuracy: {tacc:.4f} ({tacc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(tlabel, tpred, target_names=class_names, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(tlabel, tpred))

### 6.6 可视化分析

#### 混淆矩阵

In [ ]:
cm = confusion_matrix(tlabel, tpred)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
im1 = axes[0].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set_title('Confusion Matrix (Count)'); plt.colorbar(im1, ax=axes[0])
tm = np.arange(num_classes)
axes[0].set_xticks(tm); axes[0].set_xticklabels(class_names, rotation=45)
axes[0].set_yticks(tm); axes[0].set_yticklabels(class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center', color='white' if cm[i,j]>cm.max()/2 else 'black', fontsize=12)
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cmn = cm.astype('float')/cm.sum(axis=1)[:, np.newaxis]
im2 = axes[1].imshow(cmn, interpolation='nearest', cmap=plt.cm.Blues, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)'); plt.colorbar(im2, ax=axes[1])
axes[1].set_xticks(tm); axes[1].set_xticklabels(class_names, rotation=45)
axes[1].set_yticks(tm); axes[1].set_yticklabels(class_names)
for i in range(cmn.shape[0]):
    for j in range(cmn.shape[1]):
        axes[1].text(j, i, f'{cmn[i,j]:.2f}', ha='center', va='center', color='white' if cmn[i,j]>0.5 else 'black', fontsize=12)
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight'); plt.close()
print('Confusion matrix saved')

#### 特征图可视化

In [ ]:
fmaps = {}
def gf(name):
    def hook(m,i,o): fmaps[name]=o.detach()
    return hook

model.eval()
model.resnet.conv1.register_forward_hook(gf('conv1'))
si,_ = next(iter(test_loader))
with torch.no_grad(): _ = model(si[:1].to(device))
fig, axes = plt.subplots(4, 8, figsize=(20, 10))
f1 = fmaps['conv1'][0]
for i in range(min(32, f1.shape[0])):
    r,c = i//8, i%8
    axes[r][c].imshow(f1[i].cpu().numpy(), cmap='viridis'); axes[r][c].axis('off')
plt.suptitle('Conv1 Feature Maps (first 32 channels)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_maps.png'), dpi=150, bbox_inches='tight'); plt.close()
print('Feature maps saved')

#### 错误样本分析

In [ ]:
err = np.where(tpred != tlabel)[0]
print(f'Test: {len(tlabel)} total, {len(err)} errors ({len(err)/len(tlabel)*100:.2f}%)')
ne = min(8, len(err))
if ne > 0:
    mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std_t = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8)); axes = axes.flatten()
    for i in range(ne):
        si = test_dataset.indices[err[i]]
        it, tl = full_dataset[si]
        id2 = torch.clamp(it*std_t+mean_t, 0, 1).permute(1,2,0).numpy()
        pl = tpred[err[i]]
        axes[i].imshow(id2)
        axes[i].set_title(f'True: {class_names[tl]}\nPred: {class_names[pl]}', fontsize=10, color='red')
        axes[i].axis('off')
    for j in range(ne,8): axes[j].axis('off')
    plt.suptitle('Misclassified Samples', fontsize=14); plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'error_analysis.png'), dpi=150, bbox_inches='tight'); plt.close()
    print('Error analysis saved')

#### 各类别指标对比

In [ ]:
pc = precision_score(tlabel, tpred, average=None, zero_division=0)
rc = recall_score(tlabel, tpred, average=None, zero_division=0)
fc = f1_score(tlabel, tpred, average=None, zero_division=0)
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(num_classes); w = 0.25
b1 = ax.bar(x-w, pc, w, label='Precision', color='#4A90D9')
b2 = ax.bar(x, rc, w, label='Recall', color='#E74C3C')
b3 = ax.bar(x+w, fc, w, label='F1-Score', color='#27AE60')
ax.set_title('Per-Class Precision, Recall, F1-Score', fontsize=14)
ax.set_xticks(x); ax.set_xticklabels(class_names)
ax.legend(); ax.set_ylim(0, 1.1); ax.grid(axis='y', alpha=0.3)
for bars in [b1,b2,b3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', xy=(bar.get_x()+bar.get_width()/2,h), xytext=(0,3), textcoords='offset points', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'per_class_metrics.png'), dpi=150, bbox_inches='tight'); plt.close()
print('Per-class metrics saved')

#### 模型推理演示

In [ ]:
model.eval()
dl = DataLoader(test_dataset, batch_size=8, shuffle=True)
di, dl2 = next(iter(dl))
with torch.no_grad():
    do = model(di.to(device)); dp = torch.softmax(do, dim=1); dpt = torch.argmax(dp, dim=1)
mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std_t = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
fig, axes = plt.subplots(2, 4, figsize=(20, 10)); axes = axes.flatten()
for i in range(8):
    im = torch.clamp(di[i]*std_t+mean_t, 0, 1).permute(1,2,0).numpy()
    tc = class_names[dl2[i]]; pc2 = class_names[dpt[i]]; cf = dp[i][dpt[i]].item()*100
    axes[i].imshow(im); ok = dpt[i]==dl2[i]
    axes[i].set_title(f"{'OK' if ok else 'X'} True: {tc}\nPred: {pc2} ({cf:.1f}%)", fontsize=11, color='green' if ok else 'red')
    axes[i].axis('off')
plt.suptitle('Inference Demo (Green=Correct, Red=Wrong)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'inference_demo.png'), dpi=150, bbox_inches='tight'); plt.close()
print('Inference demo saved')

## 七、总结与展望

### 7.1 工作总结

1. **数据准备**：构建了包含四大类别的垃圾分类数据集
2. **模型设计**：Baseline CNN基准模型 + ResNet18迁移学习模型    
3. **训练策略**：两阶段训练法（冻结卷积层 → 微调全部参数）
4. **实验分析**：通过混淆矩阵、特征图、错误样本分析进行了可视化分析

### 7.2 不足与展望

1. **数据准备**：使用真实垃圾分类数据集替代模拟数据
2. **模型设计**：尝试EfficientNet、Vision Transformer等先进架构
3. **训练策略**：结合ONNX Runtime / TensorRT实现端侧部署        
4. **展望**：考虑扩展为40+细粒度垃圾分类，开发智能垃圾分类终端
